In [30]:
import numpy as np
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

In [31]:
df=pd.read_csv('twcs.csv')

In [32]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [33]:
df.shape

(2811774, 7)

In [34]:
df.columns


Index(['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
       'response_tweet_id', 'in_response_to_tweet_id'],
      dtype='object')

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [36]:
df.describe()

,tweet_id,in_response_to_tweet_id
count,2.811774e+06,2.017439e+06
mean,1.504565e+06,1.463141e+06
std,8.616450e+05,8.665730e+05
min,1.000000e+00,1.000000e+00
25%,7.601652e+05,7.155105e+05
50%,1.507772e+06,1.439805e+06
75%,2.253296e+06,2.220646e+06
max,2.987950e+06,2.987950e+06


In [37]:
df.isnull().sum()

tweet_id                         0
author_id                        0
inbound                          0
created_at                       0
text                             0
response_tweet_id          1040629
in_response_to_tweet_id     794335
dtype: int64

In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
brand_accounts = (
    df[df["inbound"] == False]
    .groupby("author_id")
    .size()
    .sort_values(ascending=False)
)

print("Top support accounts:")
display(brand_accounts.head(30))

Top support accounts:


author_id
AmazonHelp         169840
AppleSupport       106860
Uber_Support        56270
SpotifyCares        43265
Delta               42253
Tesco               38573
AmericanAir         36764
TMobileHelp         34317
comcastcares        33031
British_Airways     29361
SouthwestAir        28977
VirginTrains        27817
Ask_Spectrum        25860
XboxSupport         24557
sprintcare          22381
hulu_support        21872
sainsburys          19466
GWRHelp             19364
AskPlayStation      19098
ChipotleTweets      18749
VerizonSupport      17966
UPSHelp             17817
ATVIAssist          17650
O2                  16212
Safaricom_Care      16077
idea_cares          15724
AskTarget           13218
AirAsiaSupport      12829
BofA_Help           12683
SW_Help             12231
dtype: int64

In [40]:
support_accounts = [
    "AmazonHelp", "AppleSupport", "Uber_Support", "SpotifyCares",
    "Delta", "Tesco", "AmericanAir", "TMobileHelp", "comcastcares",
    "British_Airways", "SouthwestAir", "VirginTrains", "Ask_Spectrum",
    "XboxSupport", "sprintcare", "hulu_support", "sainsburys",
    "GWRHelp", "AskPlayStation", "ChipotleTweets", "VerizonSupport",
    "UPSHelp", "ATVIAssist", "O2", "Safaricom_Care", "idea_cares",
    "AskTarget", "AirAsiaSupport", "BofA_Help", "SW_Help"
]

support_df = df[
    (df["inbound"] == False) &
    (df["author_id"].isin(support_accounts))
].copy()

results = []

for brand in support_accounts:
    brand_df = support_df[support_df["author_id"] == brand]

    results.append({
        "Brand": brand,
        "Support_Replies": len(brand_df),
        "Unique_Customers": brand_df["in_response_to_tweet_id"].nunique(),
        "Replies_With_Parent": brand_df["in_response_to_tweet_id"].notna().sum()
    })

brand_ranking = pd.DataFrame(results)
brand_ranking = brand_ranking.sort_values(
    "Unique_Customers",
    ascending=False
)

display(brand_ranking.head(15))

,Brand,Support_Replies,Unique_Customers,Replies_With_Parent
0,AmazonHelp,169840,155445,169287
1,AppleSupport,106860,106696,106719
2,Uber_Support,56270,55283,56261
3,SpotifyCares,43265,41734,43243
6,AmericanAir,36764,36524,36598
4,Delta,42253,36215,42197
7,TMobileHelp,34317,33909,34287
8,comcastcares,33031,30455,33007
10,SouthwestAir,28977,28346,28889
11,VirginTrains,27817,26373,27522


In [41]:
top_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "SouthwestAir",
    "VirginTrains"
]

rows = []

for brand in top_brands:
    support = df[
        (df["author_id"] == brand) &
        (df["inbound"] == False)
    ]

    customer_ids = support["in_response_to_tweet_id"].dropna()

    customer_messages = df[
        df["tweet_id"].isin(customer_ids)
    ]

    rows.append({
        "Brand": brand,
        "Support Replies": len(support),
        "Customer Messages": len(customer_messages),
        "Unique Customers": customer_messages["author_id"].nunique(),
        "Avg Customer Text Length": round(
            customer_messages["text"].astype(str).str.len().mean(), 1
        )
    })

brand_comparison = pd.DataFrame(rows)

display(
    brand_comparison.sort_values(
        "Customer Messages",
        ascending=False
    ).reset_index(drop=True)
)

,Brand,Support Replies,Customer Messages,Unique Customers,Avg Customer Text Length
0,AmazonHelp,169840,154985,71049,116.2
1,AppleSupport,106860,106625,76366,109.3
2,Uber_Support,56270,55215,38300,120.1
3,SpotifyCares,43265,41697,27794,103.8
4,AmericanAir,36764,36457,21686,123.4
5,Delta,42253,36168,22331,110.3
6,TMobileHelp,34317,33851,19943,110.9
7,comcastcares,33031,30423,21824,111.3
8,SouthwestAir,28977,28320,19713,115.5
9,VirginTrains,27817,26321,12631,115.2


In [42]:
apple = df[
    (df["author_id"] == "AppleSupport") &
    (df["inbound"] == False) &
    (df["in_response_to_tweet_id"].notna())
].copy()

customer_ids = apple["in_response_to_tweet_id"].astype(int)

customers = df[
    df["tweet_id"].isin(customer_ids)
].copy()

conversations = customers.merge(
    apple,
    left_on="tweet_id",
    right_on="in_response_to_tweet_id",
    suffixes=("_customer", "_support")
)

print("AppleSupport conversation pairs:", len(conversations))

display(
    conversations[
        ["text_customer", "text_support"]
    ].head(10)
)


AppleSupport conversation pairs: 106648


,text_customer,text_support
0,@AppleSupport The newest update. I️ made sure ...,@115854 Lets take a closer look into this issu...
1,@AppleSupport https://t.co/NV0yucs0lB,@115854 We're here for you. Which version of t...
2,@AppleSupport Tried resetting my settings .. r...,@115855 Let's go to DM for the next steps. DM ...
3,@AppleSupport This is what it looks like https...,@115855 Any steps tried since it started last ...
4,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 That's great it has iOS 11.1 as we can...
5,@AppleSupport I️ need answers because it’s ann...,@115855 We'd like to look into this with you. ...
6,Hey @AppleSupport and anyone else who upgraded...,"@115856 Hey, let's work together to figure out..."
7,@AppleSupport This is what is happening... htt...,@115857 We'd like to investigate further with ...
8,Tf is wrong with my keyboard @115858,"@115857 Fill us in on what is happening, then ..."
9,@AppleSupport are the call centres closed for ...,@115859 We've received your DM and will contin...


In [43]:
from collections import Counter
import re

texts = conversations["text_customer"].dropna().astype(str)

keywords = {
    "Account / Login": r"\b(account|login|log in|password|apple id|id)\b",
    "Payment / Billing": r"\b(payment|pay|paid|charge|charged|billing|bill|purchase|refund)\b",
    "App / Software": r"\b(app|application|ios|update|upgrade|software|bug|crash|error)\b",
    "Device / Hardware": r"\b(iphone|ipad|mac|macbook|keyboard|screen|battery|device|hardware)\b",
    "Connectivity": r"\b(wifi|wi-fi|internet|bluetooth|network|connection|connect)\b",
    "Calls / Messages": r"\b(call|calling|phone|message|imessage|sms|facetime)\b",
    "Media / Services": r"\b(music|itunes|icloud|tv|store|podcast|subscription)\b",
    "Other": r".*"
}

intent_counts = {}

for intent, pattern in keywords.items():
    if intent == "Other":
        continue

    matches = texts.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    intent_counts[intent] = matches.sum()

intent_table = (
    pd.DataFrame(
        list(intent_counts.items()),
        columns=["Intent", "Matching_Messages"]
    )
    .sort_values("Matching_Messages", ascending=False)
)

display(intent_table)

,Intent,Matching_Messages
2,App / Software,33818
3,Device / Hardware,32840
5,Calls / Messages,22532
6,Media / Services,8284
4,Connectivity,4416
1,Payment / Billing,3614
0,Account / Login,2802


In [45]:
patterns = {
    "App / Software": r"\b(app|application|ios|update|upgrade|software|bug|crash|error)\b",
    "Device / Hardware": r"\b(iphone|ipad|mac|macbook|keyboard|screen|battery|device|hardware)\b",
    "Calls / Messages": r"\b(call|calling|phone|message|imessage|sms|facetime)\b",
    "Media / Services": r"\b(music|itunes|icloud|tv|store|podcast|subscription)\b",
    "Connectivity": r"\b(wifi|wi-fi|internet|bluetooth|network|connection|connect)\b",
    "Payment / Billing": r"\b(payment|pay|paid|charge|charged|billing|bill|purchase|refund)\b",
    "Account / Login": r"\b(account|login|log in|password|apple id|id)\b"
}

for intent, pattern in patterns.items():
    matches = conversations[
        conversations["text_customer"]
        .astype(str)
        .str.contains(pattern, case=False, regex=True, na=False)
    ]

    print(f"\n{'=' * 60}")
    print(f"{intent} — {len(matches):,} examples")
    print("=" * 60)

    display(
        matches[["text_customer", "text_support"]]
        .head(5)
    )


App / Software — 33,818 examples


,text_customer,text_support
0,@AppleSupport The newest update. I️ made sure ...,@115854 Lets take a closer look into this issu...
6,Hey @AppleSupport and anyone else who upgraded...,"@115856 Hey, let's work together to figure out..."
14,@AppleSupport I have the iPhone 6s Plus and ju...,"@115864 To make sure, is iOS 11.1 installed on..."
16,@AppleSupport iOS 11.0.3,@115865 Let's check Settings &gt; General &gt;...
19,@AppleSupport I need the software update urgen...,@115865 Hi there! What type of device are we w...



Device / Hardware — 32,840 examples


,text_customer,text_support
4,@AppleSupport I️ have an iPhone 7 Plus and yes...,@115855 That's great it has iOS 11.1 as we can...
8,Tf is wrong with my keyboard @115858,"@115857 Fill us in on what is happening, then ..."
14,@AppleSupport I have the iPhone 6s Plus and ju...,"@115864 To make sure, is iOS 11.1 installed on..."
18,@AppleSupport iPhone 7 Plus 😊,@115865 Thanks! Which iOS version is currently...
19,@AppleSupport I need the software update urgen...,@115865 Hi there! What type of device are we w...



Calls / Messages — 22,532 examples


...

,text_customer,text_support
12,"Hello, internet. Can someone explain why this ...",@115861 You're in the right place; we'll do al...
44,@AppleSupport epıl bana yardımcı olur musun te...,@116106 We offer support via Twitter in Englis...
62,@AppleSupport still no reliable Bluetooth on m...,@116341 Thanks for reaching out. DM which Blue...
93,@AppleSupport I can’t download songs. Progress...,@116855 We want to help. Does restarting help ...
104,@AppleSupport I've updated to iOS 11.1 just no...,@116855 Sounds good. Let us know the results i...



Payment / Billing — 3,614 examples


,text_customer,text_support
56,Question- @249 @115858 my iPhone6 dies very qu...,@116337 We know how important your battery is ...
107,Series1 #Applewatch would consistently get 18 ...,@116863 We want your Apple Watch to keep up wi...
170,"@AppleSupport I bought 2 iPhones X on Friday, ...",@117510 Thanks for reaching out to us. Please ...
383,@AppleSupport the screen on our iPad Pro has s...,@120215 Thank you for reaching out to us. We'd...
394,@AppleSupport Trying to download an app to a f...,@121003 We can help. As long as this app suppo...



Account / Login — 2,802 examples


,text_customer,text_support
35,"@AppleSupport Hello, I need some help regardin...",@116100 This article should help with that: ht...
132,@AppleSupport how long does it take usually fo...,@117154 Having access to your Apple ID is impo...
134,@AppleSupport My wife made the mistake of upda...,@117177 You won't be able to revert the system...
139,@AppleSupport Hello there i need help with the...,@117454 Hi! We're happy to help. Join us in DM...
154,@115858 suck! Upgrade phone &amp; I lose my Ap...,@117484 We're happy you reached out so we can ...


# 🤖 Unsupervised Learning

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

texts = conversations["text_customer"].dropna().astype(str)

sample = texts.sample(
    n=min(20000, len(texts)),
    random_state=42
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5
)

X = vectorizer.fit_transform(sample)

k = 8

model = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

clusters = model.fit_predict(X)

terms = vectorizer.get_feature_names_out()

for i in range(k):
    center = model.cluster_centers_[i]
    top_indices = center.argsort()[-10:][::-1]
    top_words = [terms[j] for j in top_indices]

    print(f"\n{'=' * 60}")
    print(f"CLUSTER {i + 1}")
    print(f"Top terms: {', '.join(top_words)}")
    print("=" * 60)

    cluster_examples = sample.iloc[
        [j for j, c in enumerate(clusters) if c == i]
    ]

    for text in cluster_examples.head(5):
        print("-", text[:250])


CLUSTER 1
Top terms: new, update, new update, 115858, phone, new iphone, iphone, applesupport, new ios, applesupport new
- Why does my @118721 create a brand new library and erase my old one every time I open it? What’s going on???? @115948 @115858 @AppleSupport
- @AppleSupport why does my new laptop have a charge thats supposed to last 10 hours but it lasts 2 if i'm lucky?
- @AppleSupport The new update won’t let me call anyone else with an updated iPhone. I have a iPhone 6s. No one can call me either. WTH
- Good morning @AppleSupport @115858 your new software update is awful my iphone 7 is restarting since yesterday night 💔 how can i fix it
- I only get audible text notifications SOMETIMES. it'll be sitting right here and I happen to look and I got 4 new texts. never a ding. what's up with that @115858??

CLUSTER 2
Top terms: applesupport, 115858, https, apple, help, just, update, app, thanks, 115858 applesupport
- @AppleSupport And the fact that there’s things out of order right no

In [47]:
def trivial_baseline(text):
    return {
        "decision": "ESCALATE",
        "reason": "Trivial baseline always sends the message to a human."
    }

examples = conversations["text_customer"].head(10)

for text in examples:
    result = trivial_baseline(text)

    print("Customer:", text[:150])
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print()

Customer: @AppleSupport The newest update. I️ made sure to download it yesterday.
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: @AppleSupport  https://t.co/NV0yucs0lB
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: @AppleSupport Tried resetting my settings .. restarting my phone .. all that
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: @AppleSupport This is what it looks like https://t.co/XCQU2l4xUB
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: @AppleSupport I️ have an iPhone 7 Plus and yes I️ do
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: @AppleSupport I️ need answers because it’s annoying 🙃
Decision: ESCALATE
Reason: Trivial baseline always sends the message to a human.

Customer: Hey @AppleSupport and anyone else who upgraded to ios11.1, are y’all having is

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

data = conversations[
    ["text_customer", "text_support"]
].dropna().drop_duplicates().reset_index(drop=True)

train = data.sample(
    n=min(50000, len(data)),
    random_state=42
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_train = vectorizer.fit_transform(
    train["text_customer"].astype(str)
)

def simple_baseline(message):
    query = vectorizer.transform([str(message)])
    scores = cosine_similarity(query, X_train)[0]
    best_index = np.argmax(scores)

    return {
        "reply": train.iloc[best_index]["text_support"],
        "similarity": float(scores[best_index])
    }

examples = data.sample(
    n=min(10, len(data)),
    random_state=100
)

for _, row in examples.iterrows():
    result = simple_baseline(row["text_customer"])

    print("CUSTOMER:")
    print(row["text_customer"])

    print("\nHISTORICAL REPLY:")
    print(result["reply"])

    print(f"\nSimilarity: {result['similarity']:.3f}")
    print("-" * 70)

CUSTOMER:
While i was previously working with the senior advisor named James on my recent case he just disconnected the chat session while i was typing. So unprofessional. Is this the way you treat your customers? @116333 @AppleSupport

HISTORICAL REPLY:
@708505 We want to continue to help out. Your DM has been received. Look for a response from us shortly there. Thank you.

Similarity: 1.000
----------------------------------------------------------------------
CUSTOMER:
@AppleSupport your new update sucks. Letter “i” is coming up as a weird symbol and totally messing up all of my messages. FIX THIS.

HISTORICAL REPLY:
@475272 We received your DM and you should expect a response there.

Similarity: 1.000
----------------------------------------------------------------------
CUSTOMER:
MY FACETIME BEEN BUGGING ALL DAY @115858 wtffffff????

HISTORICAL REPLY:
@121848 Here’s what you can do to work around the issue until it’s fixed in a future software update: https://t.co/xXaXeeSRt9

Simi

# model Classification Report

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

def assign_intent(text):
    text = str(text).lower()

    if any(x in text for x in [
        "battery", "charging", "charge", "drain", "battery life"
    ]):
        return "Battery / Power"

    if any(x in text for x in [
        "wifi", "wi-fi", "bluetooth", "internet",
        "network", "cellular", "signal", "connection"
    ]):
        return "Connectivity"

    if any(x in text for x in [
        "facetime", "imessage", "sms", "text message",
        "phone call", "calls", "calling", "voicemail"
    ]):
        return "Calls / Messaging"

    if any(x in text for x in [
        "apple id", "password", "account", "login",
        "logged in", "security", "phishing"
    ]):
        return "Apple ID / Security"

    if any(x in text for x in [
        "icloud", "itunes", "app store", "apple music",
        "podcast", "subscription"
    ]):
        return "Apple Services / Apps"

    if any(x in text for x in [
        "screen", "keyboard", "speaker", "camera",
        "iphone", "ipad", "macbook", "mac",
        "device", "touch"
    ]):
        return "Device / Hardware"

    if any(x in text for x in [
        "ios", "update", "software", "bug",
        "glitch", "crash", "freeze", "slow"
    ]):
        return "Software / iOS Issues"

    return "Other / Unclear"


ml_data = conversations[
    ["text_customer", "text_support"]
].dropna().drop_duplicates().copy()

ml_data["intent"] = ml_data["text_customer"].apply(assign_intent)

train_data, test_data = train_test_split(
    ml_data,
    test_size=0.2,
    random_state=42,
    stratify=ml_data["intent"]
)

classifier = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=15000,
            ngram_range=(1, 2)
        )
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        )
    )
])

classifier.fit(
    train_data["text_customer"],
    train_data["intent"]
)

predictions = classifier.predict(
    test_data["text_customer"]
)

print("Accuracy:", round(
    accuracy_score(
        test_data["intent"],
        predictions
    ),
    4
))

print("\nClassification Report:\n")

print(
    classification_report(
        test_data["intent"],
        predictions,
        zero_division=0
    )
)

Accuracy: 0.9687

Classification Report:

                       precision    recall  f1-score   support

  Apple ID / Security       0.95      0.94      0.95       484
Apple Services / Apps       0.97      0.98      0.97       877
      Battery / Power       1.00      0.96      0.98      2137
    Calls / Messaging       0.95      0.94      0.94       535
         Connectivity       0.98      0.96      0.97       784
    Device / Hardware       0.99      0.93      0.96      4906
      Other / Unclear       0.96      1.00      0.98      7758
Software / iOS Issues       0.95      0.96      0.96      3849

             accuracy                           0.97     21330
            macro avg       0.97      0.96      0.96     21330
         weighted avg       0.97      0.97      0.97     21330



In [51]:
payment_keywords = [
    "charged", "charge", "charged twice", "double charged",
    "payment", "paid", "pay", "billing", "bill",
    "refund", "refunded", "purchase", "receipt",
    "transaction", "money", "cost", "price"
]

def improved_intent(text):
    text = str(text).lower()

    if any(k in text for k in payment_keywords):
        return "Payment / Billing"

    return classifier.predict([text])[0]

def improved_support_agent(message):
    message = str(message)

    intent = improved_intent(message)

    probabilities = classifier.predict_proba([message])[0]
    confidence = float(np.max(probabilities))

    query = vectorizer.transform([message])
    scores = cosine_similarity(query, X_train)[0]

    best_index = np.argmax(scores)
    similarity = float(scores[best_index])

    historical_reply = train.iloc[best_index]["text_support"]

    sensitive = [
        "Payment / Billing",
        "Apple ID / Security"
    ]

    if intent in sensitive:
        decision = "ESCALATE"
        reason = "Sensitive account or payment issue requires human review."
    elif confidence < 0.70 or similarity < 0.30:
        decision = "ESCALATE"
        reason = "Low confidence or insufficiently similar historical case."
    else:
        decision = "AUTO-HANDLE"
        reason = "High-confidence intent with a sufficiently similar historical case."

    return {
        "intent": intent,
        "confidence": round(confidence, 3),
        "similarity": round(similarity, 3),
        "reply": historical_reply,
        "decision": decision,
        "reason": reason
    }

test_messages = [
    "My iPhone battery is draining very quickly after the update.",
    "I cannot connect my iPhone to WiFi.",
    "My Apple ID password is not working.",
    "FaceTime is not working on my phone.",
    "I was charged twice for my purchase."
]

for message in test_messages:
    result = improved_support_agent(message)

    print("=" * 70)
    print("CUSTOMER:", message)
    print("INTENT:", result["intent"])
    print("CONFIDENCE:", result["confidence"])
    print("SIMILARITY:", result["similarity"])
    print("REPLY:", result["reply"])
    print("DECISION:", result["decision"])
    print("REASON:", result["reason"])

CUSTOMER: My iPhone battery is draining very quickly after the update.
INTENT: Battery / Power
CONFIDENCE: 1.0
SIMILARITY: 0.642
REPLY: @400572 Were you able to check out those steps in the article we provided. Did they help at all?
DECISION: AUTO-HANDLE
REASON: High-confidence intent with a sufficiently similar historical case.
CUSTOMER: I cannot connect my iPhone to WiFi.
INTENT: Connectivity
CONFIDENCE: 0.999
SIMILARITY: 0.419
REPLY: @362687 Hi there! Can you give us some more details on what's going on? Does this happen on all Wi-Fi networks or just one?
DECISION: AUTO-HANDLE
REASON: High-confidence intent with a sufficiently similar historical case.
CUSTOMER: My Apple ID password is not working.
INTENT: Apple ID / Security
CONFIDENCE: 1.0
SIMILARITY: 0.8
REPLY: @130465 We'd be happy to look into this with you more in detail. Reach out to us via DM and we'll get started. https://t.co/GDrqU22YpT
DECISION: ESCALATE
REASON: Sensitive account or payment issue requires human review.
CUS

In [52]:
golden = conversations[["text_customer"]].dropna().sample(
    n=200,
    random_state=42
).reset_index(drop=True)

golden["intent"] = ""
golden["decision"] = ""

golden.to_csv("golden_eval_200.csv", index=False)

display(golden.head(20))
print("Golden evaluation examples:", len(golden))


,text_customer,intent,decision
0,@115858 @AppleSupport removed my earphone from...,,
1,@AppleSupport And the fact that there’s things...,,
2,@115858 fix this shit . Like wtf https://t.co/...,,
3,Why does my @118721 create a brand new library...,,
4,@AppleSupport I’m not sure if you guys sent th...,,
5,@AppleSupport Every time I try to open the Ap...,,
6,@AppleSupport 30 minutos y no abre la pagina!,,
7,@AppleSupport It was totally fine before I upd...,,
8,@AppleSupport Thank you!!!,,
9,@AppleSupport I updated my iPhone 6S to iOS 11...,,


Golden evaluation examples: 200


In [54]:
!pip install ipywidgets -q


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
from IPython.display import display, clear_output
import ipywidgets as widgets

golden = golden.reset_index(drop=True)

intents = [
    "Software / iOS Issues",
    "Device / Hardware Issues",
    "Battery / Power",
    "Calls / Messaging",
    "Apple Services / Apps",
    "Connectivity",
    "Apple ID / Security",
    "Payment / Billing",
    "Other / Unclear"
]

decisions = ["AUTO-HANDLE", "ESCALATE"]

intent_dropdown = widgets.Dropdown(
    options=intents,
    description="Intent:"
)

decision_dropdown = widgets.Dropdown(
    options=decisions,
    description="Decision:"
)

button = widgets.Button(description="Save & Next")

output = widgets.Output()

labels = []

def show_question():
    with output:
        clear_output()
        i = len(labels)

        if i >= len(golden):
            golden["intent"] = [x[0] for x in labels]
            golden["decision"] = [x[1] for x in labels]
            golden.to_csv("golden_eval_200.csv", index=False)

            print("DONE")
            print("Saved:", len(golden), "labelled examples")
            return

        print(f"Example {i+1} / {len(golden)}")
        print()
        print(golden.iloc[i]["text_customer"])

def save_next(b):
    labels.append((
        intent_dropdown.value,
        decision_dropdown.value
    ))
    show_question()

button.on_click(save_next)

display(output)
display(intent_dropdown)
display(decision_dropdown)
display(button)

show_question()

Example 1 / 200

@115858 @AppleSupport removed my earphone from my phone and this happened... HOW IS THIS EVEN POSSIBLE?! https://t.co/er5gzYrbvt


In [56]:
for i in range(20):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


1. @115858 @AppleSupport removed my earphone from my phone and this happened... HOW IS THIS EVEN POSSIBLE?! https://t.co/er5gzYrbvt

2. @AppleSupport And the fact that there’s things out of order right now is kind of driving me nuts is there any way to fix that?

3. @115858 fix this shit . Like wtf https://t.co/sOCmJYcHXS

4. Why does my @118721 create a brand new library and erase my old one every time I open it? What’s going on???? @115948 @115858 @AppleSupport

5. @AppleSupport I’m not sure if you guys sent this to me or someone is phishing for my account info https://t.co/ByWDjZCFbq

6. @AppleSupport  Every time I try to open the App Store, I doesn’t load... Then it fails to retry... https://t.co/7mLCRZRWZR

7. @AppleSupport  30 minutos y no abre la pagina!

8. @AppleSupport It was totally fine before I updated to iOS 11

9. @AppleSupport Thank you!!!

10. @AppleSupport I updated my iPhone 6S to iOS 11 and the new Podcast App does not appear to let me view or edit my queue. Please

In [57]:
labels_1_19 = [
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Payment / Billing", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_1_19):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 1-19 labelled successfully.")

Rows 1-19 labelled successfully.


In [58]:
for i in range(19, 39):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


20. @AppleSupport @358826 I have the same problem Apple!!! Do something! Too many bugs and battery drain. Want my iPhone back

21. Ever since I updated my phone last night, it’s been running soooo slow. @115858 sort your updates out

22. It would be incredible if my $750 iPhone 7 Plus didn’t freeze every other time I open an app @AppleSupport

23. @AppleSupport This is a security issue! Why not have it not auto rejoin??

24. Good morning @AppleSupport @115858 your new software update is awful my iphone 7 is restarting since yesterday night 💔 how can i fix it

25. @116333 very disappointed in the IOS 11.0.3 update it has more bugs than the amazon rainforest . Please do something! :(

26. @115858 y’all can fix these type of things in 3 seconds “ I️ “ don’t wanna see this BULLSHIT

27. Why the fuck has everything on the iphone been trash since ios11 fix this shit my nigga @115858

28. @AppleSupport a small line across my screen will not respond to touch on my iphone 5s. how do i fix it?


In [59]:
labels_20_39 = [
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Payment / Billing", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_20_39, start=19):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 20-39 labelled successfully")

Rows 20-39 labelled successfully


In [60]:
for i in range(39, 59):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


40. @AppleSupport I called 1-800apple several times this week and received different answers from each rep.

41. When @115858 keeps charging you hbo. @6174 isn’t on! I cancelled this shit. It’s the ONLY reason I had it. 😒😒😒😒

42. MY BATTERY KEEPS DRAINING SO QUICKLY, MY PHONE KEEPS FREEZING AND SPAZZING OUT. @115858 WTF HAVE YOU DONE

43. @AppleSupport 6, it lasts about 2 hours after the update.

44. So I updated my phone and I still see that the “I” is still an issue... @115858

45. Dear @115858, your new #IOS1103 is giving me trouble to download apps. Help me? Thanks #iphoneupdate

46. @AppleSupport hi if i have a new 5k imac with a kaby lake processor and high sierra can i play netflix/youtube/amazon in 4k please?

47. @115858 dear Apple...

THE MOTHERFUCKING GLITCH RUINED THE CAPTURE FOR THE EXCELLENCE OF PICTURE THAT THIS IS. FIX THIS!

Xoxo, everyone! https://t.co/ce0qms1yh4

48. ¿Anybody else having battery leaking problems with iOS 11.0.3? Drains like crazy #ios1103 @AppleSupp

In [61]:
labels_40_58 = [
    ("Other / Unclear", "ESCALATE"),
    ("Payment / Billing", "ESCALATE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_40_58, start=39):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 40-58 labelled successfully")

Rows 40-58 labelled successfully


In [62]:
for i in range(58, 78):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


59. Hmm, Seem to have lost the Hashtag function on my new MacBookPro used to be Option key + 3 - but no more? @AppleSupport

60. Hey @115858, can we maybe fix the whole I️ glitch thing? It’s kinda hard to text without using it. Ok cool thanks for the quick chat 🤙🏼🤙🏼

61. @AppleSupport loved the update for iPhone but really disappointed in the service because my phone never has a good charge anymore !!

62. @AppleSupport how do I downgrade my iPhone SE from iOS 11 to iOS 10.3.3 ? Please help!

63. Why is it happening AGAIN..???!!!
@AppleSupport 
#iphone https://t.co/KtyVqv637a

64. @115858 just completed update. Now finger security doesn’t work. How many update will IOS11 need?

65. HELP... Question for iPhone users.
How do I backup/restore a new phone WITHOUT transferring my 10 000 photos across? 
Wana save storage🤔???

66. @115858 can y’all PLEASE fix the issue with your keyboard and specific letters turning to symbols???

67. Where did all the dings go, @AppleSupport?  #iPhone is ri

In [63]:
labels_59_78 = [
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_59_78, start=58):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 59-78 labelled successfully")

Rows 59-78 labelled successfully


In [64]:
for i in range(78, 98):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


79. @727897 @AppleSupport @115858 Get no sound when I receive a message

80. @AppleSupport 3rd party said to refer to you they couldn’t help. Most notifications fall off the badge or alert but it depends how long they are.

81. The Apple update has many cool features. It has my phone freezing, shutting off, not staying connected to WiFi, and more. Great job, @115858!

82. @AppleSupport It’s all on the Apple Support thread in the first tweet I sent over.

83. @AppleSupport #ayuda #porfavor en el arranque allí se queda que puedo hacer?
Gracias 🙏🏼 https://t.co/ay2Vo2ALE0

84. @AppleSupport Taking portrait mode photos keeps crashing and restarting my iPhone X. Other modes are fine. Already on iOS 11.1. Help!

85. @AppleSupport my iPhone keeps bleeping like a fire alarm, Ive closed all my apps and webpages and internet history, how do I stop this noise

86. @AppleSupport we can't accept toc on mobile....this disrupted work today

87. @AppleSupport I pulled these out of my phone and the lig

In [66]:
labels_79_97 = [
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE")
]

for i, (intent, decision) in enumerate(labels_79_97, start=78):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 79-97 labelled successfully")

Rows 79-97 labelled successfully


In [67]:
for i in range(97, 117):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


98. @115858 what’s the deal with this update? My iPhone7️⃣ picks and chooses when the touch screen wants to work. 🚮

99. @AppleSupport  https://t.co/8Ys92KOmQ5

100. Unless Im doing something wrong, Notifications doesnt dropdown in Reachability anymore 😞@AppleSupport @115858

101. Hey @AppleSupport, my 2016 MBP has been crashing at least once per day. Care to give me some insight based on a crash report?

102. @AppleSupport Iphone 6, ios 11.1.2; problem is very annoying, keyboard is also problematic, vry slow

103. @AppleSupport why did you delete some of my contacts

104. @AppleSupport hi 😊! I have some issues with my controle center on my iPhone 8 plus

105. Why is the letter I️ not working. Messed up my tweet @115858 https://t.co/4ewUIeHpwc

106. Why won’t my phone back up @115858 ?

107. @AppleSupport why is it so hard to display the letter “I” in #iOS11 #yougotthis https://t.co/4UzMP4U3gz

108. @115858 can you explain to me why these boxes pop up when I️ type “I️” and can y’all f

In [68]:
labels_98_117 = [
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Device / Hardware Issues", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_98_117, start=97):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 98-117 labelled successfully")

Rows 98-117 labelled successfully


In [69]:
for i in range(117, 137):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


118. @AppleSupport @115858 why do my letter i️’s show up as a box. i️ want ya to fix it pretty plz and thx

119. @AppleSupport I had to get our IT people on it to fix it. Well they were only able to create a work around for me. But so far it has been ok.

120. @AppleSupport why does your new OS not work on the iPhone 6?!?!

121. @AppleSupport it’s imported

122. @AppleSupport 11.0.03 has made my IPh6 useless, bluetooth issues, battery life, frozen screen, call answering problems. Headed to Android

123. @AppleSupport Could you please fix this bug in ‘Kannada’ language keyboard. Character gets jumbled up.
#ಕನ್ನಡರಾಜ್ಯೋತ್ಸವ 
#ಕನ್ನಡ https://t.co/iR39sCkscp

124. @486429 @115858 I’m frustrated 😫😫

125. Irrelevant glitch or scary failure of Find My Mac? Why do I see a stranger’s computer in my account, @AppleSupport? https://t.co/dEfPsvISYv

126. @AppleSupport It show 11.0.3  Not only does it use battery life at a rapid rate but the screen comes up sideways and at times upside down.

127. @

In [70]:
labels_118_137 = [
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Payment / Billing", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_118_137, start=117):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 118-137 labelled successfully")

Rows 118-137 labelled successfully


In [71]:
for i in range(137, 157):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


138. Recién hoy actualicé el iOS. MALÍSIMO. Entre otros bugs horribles, pones modo avión desde el acceso directo , lo sacas, y no te vuelven los datos automático, tenes q ir ajustes. Steve Jobs está muriendo muerto. @115858

139. @115858 IOS 11 has completely killed my phone, every app freezes and closes which is almost as annoying as being priced out of a new one.

140. @115858 I️ recently just updated and my I️ ‘s are weird... uhhh

141. @AppleSupport @342946 the iOS11/Watch 4 updates have not played nice with my #starkeyhalos #applewatch. Bad connectivity.

142. @AppleSupport @115858 i refuse to pay for a upgrade so either fix this or run me my money or a FREE upgraded phone! 🤬😭😂😂😂 https://t.co/LOfQHctLk6

143. @AppleSupport Yeah, didn’t work. Software update?

144. I️ blame @115858 for my typo I️ was over here trying to figure out how to say it without using “I️” and look where it got me fuck you apple !

145. LOOK @115858 ... fix this annoying I️ shit 😒

146. @AppleSupport @11585

In [72]:
labels_138_157 = [
    ("Connectivity", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Payment / Billing", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Payment / Billing", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Connectivity", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Battery / Power", "AUTO-HANDLE")
]

for i, (intent, decision) in enumerate(labels_138_157, start=137):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 138-157 labelled successfully")

Rows 138-157 labelled successfully


In [73]:
for i in range(157, 177):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


158. @AppleSupport the iPhone restarted on its own and deleted my photo data. What’s going on with that?

159. @AppleSupport @509571 I did thisssss it still don’t work

160. @AppleSupport Not cool... ☹️ https://t.co/t1nd4Y7q5Y

161. @AppleSupport I have an iPhone 6s. It’s running iOS 11.1.1. Already restarted it.

162. @AppleSupport sir please tell me how much Year guarantee of iPad

163. @AppleSupport my phone keeps freezing on the Apple logo for 2+ hours at a time &amp; won’t allow me to make or receive calls...

164. When y’all fixing Apple Music @115858?

165. Wish @115858 would fix this I️ problem cause it’s annoying 🤗🤗

166. @AppleSupport What's happening here 👇👇👇 https://t.co/WrwXCXJHPn

167. @AppleSupport I did the new Up now I can’t text certain phone numbers?

168. @AppleSupport Yes. I’ve just upgraded to IoS11.2 and it seems to be fixed!

169. @118721 music, @115858 I have several Eminem songs that I would like to be reimbursed for , please remove his music and give me a re

In [74]:
labels_158_177 = [
    ("Software / iOS Issues", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Payment / Billing", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE")
]

for i, (intent, decision) in enumerate(labels_158_177, start=157):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 158-177 labelled successfully")

Rows 158-177 labelled successfully


In [75]:
for i in range(177, 200):
    print(f"\n{i+1}. {golden.iloc[i]['text_customer']}")


178. @AppleSupport My SE phone goes to the black screen with spinning wheel and asks for touch id or passcode once in a while when using the phone. How do i fix it?

179. Oh great, iMessage not working after the latest iOS update. 😐 @AppleSupport

180. @574435 @AppleSupport I too @78108 terrible battery performance issues with ios 11. Wish i had stayed on 10.3.3. All my devices. Iphone SE, 6s plus and ipad pro

181. @AppleSupport 11.1.1 (15B150)

182. @AppleSupport I’ve tried wall outlet and docking station. Get this very briefly when holding down power button then it’s blank again https://t.co/pqGJXQrrGo

183. @115858 Ur new ios11 is so stupid and worst ever update that now my 10 month battery doesn’t even last for 12 hrs. Salute to ur testers.

184. So why are all I’s looking like this?
And quotation marks are « « 
@TwitterSupport @AppleSupport

185. @AppleSupport chicos cuando van a arreglar ios11.0.3? Es súper inestable.Lags en escritura,poca duración de batería,lags en el menú en

In [76]:
labels_178_200 = [
    ("Software / iOS Issues", "ESCALATE"),
    ("Calls / Messaging", "AUTO-HANDLE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Device / Hardware Issues", "ESCALATE"),
    ("Battery / Power", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Apple Services / Apps", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Battery / Power", "ESCALATE"),
    ("Other / Unclear", "AUTO-HANDLE"),
    ("Apple Services / Apps", "AUTO-HANDLE"),
    ("Other / Unclear", "ESCALATE"),
    ("Software / iOS Issues", "AUTO-HANDLE"),
    ("Software / iOS Issues", "ESCALATE"),
    ("Apple ID / Security", "ESCALATE"),
    ("Software / iOS Issues", "ESCALATE")
]

for i, (intent, decision) in enumerate(labels_178_200, start=177):
    golden.loc[i, "intent"] = intent
    golden.loc[i, "decision"] = decision

print("Rows 178-200 labelled successfully")
print("Golden set completed: 200 rows")

Rows 178-200 labelled successfully
Golden set completed: 200 rows


In [77]:
golden.to_csv("golden_eval_200.csv", index=False)

print("Saved: golden_eval_200.csv")

Saved: golden_eval_200.csv


In [78]:
from sklearn.metrics import accuracy_score, f1_score

golden["predicted_intent"] = golden["text_customer"].apply(
    improved_intent
)

intent_accuracy = accuracy_score(
    golden["intent"],
    golden["predicted_intent"]
)

intent_macro_f1 = f1_score(
    golden["intent"],
    golden["predicted_intent"],
    average="macro"
)

print("Intent Accuracy:", round(intent_accuracy, 3))
print("Intent Macro F1:", round(intent_macro_f1, 3))

Intent Accuracy: 0.385
Intent Macro F1: 0.355


In [79]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print(classification_report(
    golden["intent"],
    golden["predicted_intent"],
    zero_division=0
))

cm = pd.crosstab(
    golden["intent"],
    golden["predicted_intent"]
)

display(cm)

                          precision    recall  f1-score   support

     Apple ID / Security       0.60      0.43      0.50         7
   Apple Services / Apps       0.83      0.23      0.36        22
         Battery / Power       0.62      0.62      0.62        16
       Calls / Messaging       0.40      0.22      0.29         9
            Connectivity       0.60      0.33      0.43         9
       Device / Hardware       0.00      0.00      0.00         0
Device / Hardware Issues       0.00      0.00      0.00        23
         Other / Unclear       0.35      0.87      0.50        31
       Payment / Billing       0.33      0.57      0.42         7
   Software / iOS Issues       0.74      0.30      0.43        76

                accuracy                           0.39       200
               macro avg       0.45      0.36      0.35       200
            weighted avg       0.56      0.39      0.39       200



predicted_intent,Apple ID / Security,Apple Services / Apps,Battery / Power,Calls / Messaging,Connectivity,Device / Hardware,Other / Unclear,Payment / Billing,Software / iOS Issues
intent,,,,,,,,,
Apple ID / Security,3,0,0,0,0,0,3,0,1
Apple Services / Apps,0,5,0,0,0,7,7,0,3
Battery / Power,0,0,10,0,0,0,1,4,1
Calls / Messaging,0,0,0,2,1,2,4,0,0
Connectivity,1,0,1,0,3,0,1,1,2
Device / Hardware Issues,0,0,1,0,0,10,10,1,1
Other / Unclear,0,0,1,0,0,2,27,1,0
Payment / Billing,1,0,1,0,0,0,1,4,0
Software / iOS Issues,0,1,2,3,1,22,23,1,23


In [80]:
id2 = "improved_classifier"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def better_intent(text):
    text = str(text).lower()

    if any(k in text for k in [
        "charged", "charge", "payment", "paid", "billing",
        "bill", "refund", "purchase", "receipt", "transaction",
        "money", "cost", "price"
    ]):
        return "Payment / Billing"

    if any(k in text for k in [
        "apple id", "password", "account", "login", "logged in",
        "security", "phishing", "verification", "contacts missing"
    ]):
        return "Apple ID / Security"

    if any(k in text for k in [
        "battery", "charging", "charge", "drain", "battery life",
        "won't hold a charge", "charge to"
    ]):
        return "Battery / Power"

    if any(k in text for k in [
        "wifi", "wi-fi", "bluetooth", "internet", "network",
        "cellular", "signal", "connection", "connectivity"
    ]):
        return "Connectivity"

    if any(k in text for k in [
        "facetime", "imessage", "sms", "text", "texting",
        "message", "messages", "call", "calling", "voicemail"
    ]):
        return "Calls / Messaging"

    if any(k in text for k in [
        "icloud", "itunes", "app store", "apple music",
        "podcast", "subscription", "airplay", "app"
    ]):
        return "Apple Services / Apps"

    if any(k in text for k in [
        "screen", "keyboard", "speaker", "camera", "touchscreen",
        "iphone", "ipad", "macbook", "mac", "charger",
        "charging port", "device", "phone"
    ]):
        return "Device / Hardware Issues"

    if any(k in text for k in [
        "ios", "update", "software", "bug", "glitch",
        "freeze", "freezing", "crash", "crashing", "slow",
        "restart", "restarted", "autocorrect", "spelling"
    ]):
        return "Software / iOS Issues"

    return "Other / Unclear"


golden["predicted_intent"] = golden["text_customer"].apply(better_intent)

intent_accuracy = accuracy_score(
    golden["intent"],
    golden["predicted_intent"]
)

intent_macro_f1 = f1_score(
    golden["intent"],
    golden["predicted_intent"],
    average="macro"
)

print("Improved Intent Accuracy:", round(intent_accuracy, 3))
print("Improved Intent Macro F1:", round(intent_macro_f1, 3))


Improved Intent Accuracy: 0.28
Improved Intent Macro F1: 0.368


In [81]:

golden_texts = set(golden["text_customer"].astype(str))

clean_data = data[
    ~data["text_customer"].astype(str).isin(golden_texts)
].copy()

print("Original training examples:", len(data))
print("After removing golden examples:", len(clean_data))

# Use the remaining historical conversations
train_clean = clean_data.sample(
    n=min(50000, len(clean_data)),
    random_state=42
).reset_index(drop=True)

# TF-IDF features
clean_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

X_clean = clean_vectorizer.fit_transform(
    train_clean["text_customer"].astype(str)
)

# Create training labels from the improved intent rules
y_clean = train_clean["text_customer"].apply(better_intent)

# Train classifier
clean_classifier = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clean_classifier.fit(X_clean, y_clean)

# Predict golden set
golden["predicted_intent"] = clean_classifier.predict(
    clean_vectorizer.transform(
        golden["text_customer"].astype(str)
    )
)

intent_accuracy = accuracy_score(
    golden["intent"],
    golden["predicted_intent"]
)

intent_macro_f1 = f1_score(
    golden["intent"],
    golden["predicted_intent"],
    average="macro"
)

print("Clean Intent Accuracy:", round(intent_accuracy, 3))
print("Clean Intent Macro F1:", round(intent_macro_f1, 3))

Original training examples: 106648
After removing golden examples: 106442
Clean Intent Accuracy: 0.28
Clean Intent Macro F1: 0.358


In [82]:
id2 = "eval_decision"

def predict_decision(message):
    result = improved_support_agent(message)
    return result["decision"]

golden["predicted_decision"] = golden["text_customer"].apply(
    predict_decision
)

decision_accuracy = accuracy_score(
    golden["decision"],
    golden["predicted_decision"]
)

decision_macro_f1 = f1_score(
    golden["decision"],
    golden["predicted_decision"],
    average="macro"
)

print("Decision Accuracy:", round(decision_accuracy, 3))
print("Decision Macro F1:", round(decision_macro_f1, 3))

Decision Accuracy: 0.625
Decision Macro F1: 0.551


In [83]:
from sklearn.metrics import classification_report

print(classification_report(
    golden["decision"],
    golden["predicted_decision"],
    zero_division=0
))

errors = golden[
    golden["decision"] != golden["predicted_decision"]
][["text_customer", "intent", "decision", "predicted_decision"]]

print("Routing errors:", len(errors))
display(errors.head(20))

              precision    recall  f1-score   support

 AUTO-HANDLE       0.69      0.79      0.73       131
    ESCALATE       0.44      0.32      0.37        69

    accuracy                           0.62       200
   macro avg       0.56      0.55      0.55       200
weighted avg       0.60      0.62      0.61       200

Routing errors: 75


,text_customer,intent,decision,predicted_decision
2,@115858 fix this shit . Like wtf https://t.co/...,Other / Unclear,ESCALATE,AUTO-HANDLE
3,Why does my @118721 create a brand new library...,Apple Services / Apps,AUTO-HANDLE,ESCALATE
10,"Dear @115858, please fix this I️",Other / Unclear,ESCALATE,AUTO-HANDLE
12,Getting sick of this @AppleSupport @191435 htt...,Other / Unclear,ESCALATE,AUTO-HANDLE
14,@AppleSupport why does my new laptop have a ch...,Battery / Power,AUTO-HANDLE,ESCALATE
22,@AppleSupport This is a security issue! Why no...,Connectivity,AUTO-HANDLE,ESCALATE
25,@115858 y’all can fix these type of things in ...,Other / Unclear,ESCALATE,AUTO-HANDLE
26,Why the fuck has everything on the iphone been...,Software / iOS Issues,ESCALATE,AUTO-HANDLE
29,@AppleSupport Why does the Health app on iOS 1...,Apple Services / Apps,AUTO-HANDLE,ESCALATE
30,anyone having issues with #osx #HighSierra and...,Software / iOS Issues,AUTO-HANDLE,ESCALATE


In [84]:
def improved_decision(message):
    message = str(message)

    intent = better_intent(message)

    query = clean_vectorizer.transform([message])
    scores = cosine_similarity(query, X_clean)[0]
    similarity = float(np.max(scores))

    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        return "ESCALATE"

    if similarity < 0.30:
        return "ESCALATE"

    return "AUTO-HANDLE"


golden["predicted_decision"] = golden["text_customer"].apply(
    improved_decision
)

decision_accuracy = accuracy_score(
    golden["decision"],
    golden["predicted_decision"]
)

decision_macro_f1 = f1_score(
    golden["decision"],
    golden["predicted_decision"],
    average="macro"
)

print("Improved Decision Accuracy:", round(decision_accuracy, 3))
print("Improved Decision Macro F1:", round(decision_macro_f1, 3))

Improved Decision Accuracy: 0.68
Improved Decision Macro F1: 0.579


In [85]:
def evaluate_agent(message):
    result = improved_support_agent(message)

    return pd.Series({
        "predicted_intent": result["intent"],
        "confidence": result["confidence"],
        "similarity": result["similarity"],
        "reply": result["reply"],
        "predicted_decision": result["decision"],
        "reason": result["reason"]
    })

agent_results = golden["text_customer"].apply(evaluate_agent)

golden_eval = pd.concat(
    [golden, agent_results],
    axis=1
)

print("Agent evaluation completed:", len(golden_eval), "examples")

Agent evaluation completed: 200 examples


In [86]:
display(
    golden_eval[
        [
            "text_customer",
            "intent",
            "predicted_intent",
            "similarity",
            "reply",
            "decision",
            "predicted_decision",
            "reason"
        ]
    ].head(10)
)

,text_customer,intent,predicted_intent,predicted_intent,similarity,reply,decision,predicted_decision,predicted_decision,reason
0,@115858 @AppleSupport removed my earphone from...,Device / Hardware Issues,Apple Services / Apps,Other / Unclear,1.0,@515507 Let us know in DM which device you're ...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
1,@AppleSupport And the fact that there’s things...,Software / iOS Issues,Apple Services / Apps,Other / Unclear,1.0,@699079 We'd like to take a look into this. Pl...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
2,@115858 fix this shit . Like wtf https://t.co/...,Other / Unclear,Other / Unclear,Other / Unclear,1.0,@170457 We want to help you with this. We see ...,ESCALATE,ESCALATE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
3,Why does my @118721 create a brand new library...,Apple Services / Apps,Apple Services / Apps,Other / Unclear,1.0,@814563 We'd be happy to get to the bottom of ...,AUTO-HANDLE,AUTO-HANDLE,ESCALATE,Low confidence or insufficiently similar histo...
4,@AppleSupport I’m not sure if you guys sent th...,Apple ID / Security,Apple ID / Security,Apple ID / Security,1.0,@185385 We'd like to help. That is not a genui...,ESCALATE,ESCALATE,ESCALATE,Sensitive account or payment issue requires hu...
5,@AppleSupport Every time I try to open the Ap...,Apple Services / Apps,Apple Services / Apps,Apple Services / Apps,1.0,@774812 Thanks for bringing this to our attent...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
6,@AppleSupport 30 minutos y no abre la pagina!,Apple Services / Apps,Apple Services / Apps,Other / Unclear,1.0,@143062 We offer support via Twitter in Englis...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
7,@AppleSupport It was totally fine before I upd...,Software / iOS Issues,Apple Services / Apps,Software / iOS Issues,1.0,@646854 If you go to Settings &gt; General &gt...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
8,@AppleSupport Thank you!!!,Other / Unclear,Apple Services / Apps,Other / Unclear,1.0,@223560 You're welcome! Enjoy the upcoming wee...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...
9,@AppleSupport I updated my iPhone 6S to iOS 11...,Apple Services / Apps,Apple Services / Apps,Apple Services / Apps,1.0,@474973 We'd love to offer our help. Are you r...,AUTO-HANDLE,AUTO-HANDLE,AUTO-HANDLE,High-confidence intent with a sufficiently sim...


In [87]:
def clean_support_agent(message):
    message = str(message)

    # Intent
    intent = clean_classifier.predict(
        clean_vectorizer.transform([message])
    )[0]

    # Retrieval
    query = clean_vectorizer.transform([message])
    scores = cosine_similarity(query, X_clean)[0]

    best_index = np.argmax(scores)
    similarity = float(scores[best_index])

    historical_reply = train_clean.iloc[best_index]["text_support"]

    # Risk-based routing
    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        decision = "ESCALATE"
        reason = "Sensitive or unclear issue requires human review."
    elif similarity < 0.30:
        decision = "ESCALATE"
        reason = "Insufficiently similar historical evidence."
    else:
        decision = "AUTO-HANDLE"
        reason = "Clear intent with sufficient historical evidence."

    return {
        "intent": intent,
        "similarity": round(similarity, 3),
        "reply": historical_reply,
        "decision": decision,
        "reason": reason
    }


def evaluate_clean_agent(message):
    result = clean_support_agent(message)

    return pd.Series({
        "predicted_intent": result["intent"],
        "similarity": result["similarity"],
        "reply": result["reply"],
        "predicted_decision": result["decision"],
        "reason": result["reason"]
    })


agent_results = golden["text_customer"].apply(
    evaluate_clean_agent
)

golden_eval = pd.concat(
    [golden, agent_results],
    axis=1
)

print("Clean agent evaluation completed:", len(golden_eval))

Clean agent evaluation completed: 200


In [88]:
print(
    "Intent Accuracy:",
    round(
        accuracy_score(
            golden_eval["intent"],
            golden_eval["predicted_intent"]
        ),
        3
    )
)

print(
    "Decision Accuracy:",
    round(
        accuracy_score(
            golden_eval["decision"],
            golden_eval["predicted_decision"]
        ),
        3
    )
)

ValueError: Classification metrics can't handle a mix of multiclass and multiclass-multioutput targets

In [89]:
# Remove duplicate columns
golden_eval = golden_eval.loc[:, ~golden_eval.columns.duplicated()]

# Recalculate metrics
intent_accuracy = accuracy_score(
    golden_eval["intent"],
    golden_eval["predicted_intent"]
)

decision_accuracy = accuracy_score(
    golden_eval["decision"],
    golden_eval["predicted_decision"]
)

print("Intent Accuracy:", round(intent_accuracy, 3))
print("Decision Accuracy:", round(decision_accuracy, 3))

Intent Accuracy: 0.28
Decision Accuracy: 0.68


In [90]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_top_cases(message, k=3):
    query = clean_vectorizer.transform([str(message)])
    scores = cosine_similarity(query, X_clean)[0]

    top_indices = np.argsort(scores)[-k:][::-1]

    return train_clean.iloc[top_indices][
        ["text_customer", "text_support"]
    ].copy(), scores[top_indices]


# Test on the first 5 golden examples
for i in range(5):
    cases, scores = get_top_cases(
        golden.iloc[i]["text_customer"]
    )

    print(f"\nExample {i+1}")
    print("Customer:", golden.iloc[i]["text_customer"])

    for j in range(len(cases)):
        print("\nEvidence", j+1)
        print("Similarity:", round(float(scores[j]), 3))
        print("Historical reply:", cases.iloc[j]["text_support"])


Example 1
Customer: @115858 @AppleSupport removed my earphone from my phone and this happened... HOW IS THIS EVEN POSSIBLE?! https://t.co/er5gzYrbvt

Evidence 1
Similarity: 0.456
Historical reply: @770590 Thanks for reaching out. We want to take a closer look at this with you. How long has this been going on? Send us some more details about it in DM, and we’ll work there. https://t.co/GDrqU22YpT

Evidence 2
Similarity: 0.418
Historical reply: @138707 Alright, we'd like to take a look at your options. Let us know when and where this set was purchased in a DM and we'll continue. https://t.co/GDrqU22YpT

Evidence 3
Similarity: 0.347
Historical reply: @370627 We’d be glad to look into this. Can you let us know more about the situation via DM? https://t.co/GDrqU22YpT

Example 2
Customer: @AppleSupport And the fact that there’s things out of order right now is kind of driving me nuts is there any way to fix that?

Evidence 1
Similarity: 0.553
Historical reply: @517313 Here’s what you can do

In [91]:
def evidence_score(message, k=3):
    cases, scores = get_top_cases(message, k)

    top_similarity = float(scores[0])
    avg_similarity = float(np.mean(scores))

    if top_similarity >= 0.50 and avg_similarity >= 0.40:
        evidence_quality = "STRONG"
    elif top_similarity >= 0.35:
        evidence_quality = "MODERATE"
    else:
        evidence_quality = "WEAK"

    return {
        "top_similarity": round(top_similarity, 3),
        "avg_similarity": round(avg_similarity, 3),
        "evidence_quality": evidence_quality
    }


evidence_results = golden["text_customer"].apply(
    evidence_score
).apply(pd.Series)

golden_eval = pd.concat(
    [golden_eval, evidence_results],
    axis=1
)

print(
    golden_eval["evidence_quality"].value_counts()
)

evidence_quality
MODERATE    97
STRONG      88
WEAK        15
Name: count, dtype: int64


In [92]:
display(
    golden_eval[
        [
            "text_customer",
            "intent",
            "reply",
            "top_similarity",
            "avg_similarity",
            "evidence_quality"
        ]
    ].head(10)
)

,text_customer,intent,reply,top_similarity,avg_similarity,evidence_quality
0,@115858 @AppleSupport removed my earphone from...,Device / Hardware Issues,@770590 Thanks for reaching out. We want to ta...,0.456,0.407,MODERATE
1,@AppleSupport And the fact that there’s things...,Software / iOS Issues,@517313 Here’s what you can do to work around ...,0.553,0.516,STRONG
2,@115858 fix this shit . Like wtf https://t.co/...,Other / Unclear,@285468 Let's follow up via DM to take a look ...,0.527,0.510,STRONG
3,Why does my @118721 create a brand new library...,Apple Services / Apps,"@703037 We love hearing from our customers, th...",0.415,0.352,MODERATE
4,@AppleSupport I’m not sure if you guys sent th...,Apple ID / Security,@723897 Let's look into this a little deeper. ...,0.588,0.573,STRONG
5,@AppleSupport Every time I try to open the Ap...,Apple Services / Apps,@359109 Can you please clarify what's getting ...,0.454,0.398,MODERATE
6,@AppleSupport 30 minutos y no abre la pagina!,Apple Services / Apps,@130248 Apologies for the confusion. We meant ...,0.436,0.428,MODERATE
7,@AppleSupport It was totally fine before I upd...,Software / iOS Issues,@172638 How often does the issue occur?,0.480,0.457,MODERATE
8,@AppleSupport Thank you!!!,Other / Unclear,@518834 You're very welcome. Let us know if yo...,1.000,1.000,STRONG
9,@AppleSupport I updated my iPhone 6S to iOS 11...,Apple Services / Apps,@371644 Glad to help! You can refer to the ste...,0.341,0.334,WEAK


In [93]:
import re

def clean_reply(text):
    text = str(text)

    # Remove usernames and URLs from historical replies
    text = re.sub(r'@\d+', '', text)
    text = re.sub(r'https?://\S+', '', text)

    # Clean extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def grounded_reply(message):
    cases, scores = get_top_cases(message, k=3)

    top_similarity = float(scores[0])

    if top_similarity < 0.35:
        return {
            "reply": "Thanks for reaching out. We need a few more details to help with this issue.",
            "evidence_quality": "WEAK"
        }

    historical = clean_reply(cases.iloc[0]["text_support"])

    return {
        "reply": historical,
        "evidence_quality": (
            "STRONG" if top_similarity >= 0.50
            else "MODERATE"
        )
    }


# Test on first 5 examples
for i in range(5):
    result = grounded_reply(
        golden.iloc[i]["text_customer"]
    )

    print(f"\nExample {i+1}")
    print("Customer:", golden.iloc[i]["text_customer"])
    print("Draft:", result["reply"])
    print("Evidence:", result["evidence_quality"])


Example 1
Customer: @115858 @AppleSupport removed my earphone from my phone and this happened... HOW IS THIS EVEN POSSIBLE?! https://t.co/er5gzYrbvt
Draft: Thanks for reaching out. We want to take a closer look at this with you. How long has this been going on? Send us some more details about it in DM, and we’ll work there.
Evidence: MODERATE

Example 2
Customer: @AppleSupport And the fact that there’s things out of order right now is kind of driving me nuts is there any way to fix that?
Draft: Here’s what you can do to work around the issue until it’s fixed in a future software update:
Evidence: STRONG

Example 3
Customer: @115858 fix this shit . Like wtf https://t.co/sOCmJYcHXS
Draft: Let's follow up via DM to take a look with you. Tell us the version of iOS installed in Settings &gt; General &gt; About there.
Evidence: STRONG

Example 4
Customer: Why does my @118721 create a brand new library and erase my old one every time I open it? What’s going on???? @115948 @115858 @AppleSuppo

In [94]:
def final_support_agent(message):
    message = str(message)

    # Intent
    intent = clean_classifier.predict(
        clean_vectorizer.transform([message])
    )[0]

    # Retrieve historical evidence
    cases, scores = get_top_cases(message, k=3)

    top_similarity = float(scores[0])
    avg_similarity = float(np.mean(scores))

    # Evidence quality
    if top_similarity >= 0.50 and avg_similarity >= 0.40:
        evidence_quality = "STRONG"
    elif top_similarity >= 0.35:
        evidence_quality = "MODERATE"
    else:
        evidence_quality = "WEAK"

    # Routing
    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        decision = "ESCALATE"
        reason = "Sensitive or unclear issue requires human review."

    elif evidence_quality == "WEAK":
        decision = "ESCALATE"
        reason = "Insufficient historical evidence to safely answer."

    else:
        decision = "AUTO-HANDLE"
        reason = "Clear intent with sufficient historical evidence."

    # Grounded draft
    historical_reply = clean_reply(
        cases.iloc[0]["text_support"]
    )

    if decision == "ESCALATE":
        draft = (
            "Thanks for reaching out. "
            "We'd like to take a closer look at this issue. "
            "Please provide a few more details so our support team can help."
        )
    else:
        draft = historical_reply

    return {
        "intent": intent,
        "similarity": round(top_similarity, 3),
        "evidence_quality": evidence_quality,
        "reply": draft,
        "decision": decision,
        "reason": reason
    }


# Test
for i in range(5):
    result = final_support_agent(
        golden.iloc[i]["text_customer"]
    )

    print(f"\nExample {i+1}")
    print("Intent:", result["intent"])
    print("Evidence:", result["evidence_quality"])
    print("Reply:", result["reply"])
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])


Example 1
Intent: Apple Services / Apps
Evidence: MODERATE
Reply: Thanks for reaching out. We want to take a closer look at this with you. How long has this been going on? Send us some more details about it in DM, and we’ll work there.
Decision: AUTO-HANDLE
Reason: Clear intent with sufficient historical evidence.

Example 2
Intent: Apple Services / Apps
Evidence: STRONG
Reply: Here’s what you can do to work around the issue until it’s fixed in a future software update:
Decision: AUTO-HANDLE
Reason: Clear intent with sufficient historical evidence.

Example 3
Intent: Other / Unclear
Evidence: STRONG
Reply: Thanks for reaching out. We'd like to take a closer look at this issue. Please provide a few more details so our support team can help.
Decision: ESCALATE
Reason: Sensitive or unclear issue requires human review.

Example 4
Intent: Apple Services / Apps
Evidence: MODERATE
Reply: We love hearing from our customers, that's how we improve our products. We welcome feedback here:
Decisio

In [97]:
# Final evaluation on the 200-example golden set

final_results = golden["text_customer"].apply(
    final_support_agent
).apply(pd.Series)

true_intent = golden["intent"].astype(str)
pred_intent = final_results["intent"].astype(str)

true_decision = golden["decision"].astype(str)
pred_decision = final_results["decision"].astype(str)

print("FINAL RESULTS")
print("Intent Accuracy:", round(accuracy_score(true_intent, pred_intent), 3))
print("Intent Macro F1:", round(f1_score(true_intent, pred_intent, average="macro"), 3))
print("Decision Accuracy:", round(accuracy_score(true_decision, pred_decision), 3))
print("Decision Macro F1:", round(f1_score(true_decision, pred_decision, average="macro"), 3))


FINAL RESULTS
Intent Accuracy: 0.28
Intent Macro F1: 0.358
Decision Accuracy: 0.61
Decision Macro F1: 0.531


In [98]:
def final_support_agent(message):
    message = str(message)

    # Use improved rule-based intent
    intent = better_intent(message)

    # Retrieve historical evidence
    cases, scores = get_top_cases(message, k=3)

    top_similarity = float(scores[0])
    avg_similarity = float(np.mean(scores))

    if top_similarity >= 0.50 and avg_similarity >= 0.40:
        evidence_quality = "STRONG"
    elif top_similarity >= 0.35:
        evidence_quality = "MODERATE"
    else:
        evidence_quality = "WEAK"

    # Risk-first routing
    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        decision = "ESCALATE"
        reason = "Sensitive or unclear issue requires human review."

    elif evidence_quality == "WEAK":
        decision = "ESCALATE"
        reason = "Insufficient historical evidence to safely answer."

    else:
        decision = "AUTO-HANDLE"
        reason = "Clear intent with sufficient historical evidence."

    # Grounded historical reply
    historical_reply = clean_reply(
        cases.iloc[0]["text_support"]
    )

    if decision == "ESCALATE":
        reply = (
            "Thanks for reaching out. "
            "We'd like to take a closer look at this issue. "
            "Please provide a few more details so our support team can help."
        )
    else:
        reply = historical_reply

    return {
        "intent": intent,
        "similarity": round(top_similarity, 3),
        "evidence_quality": evidence_quality,
        "reply": reply,
        "decision": decision,
        "reason": reason
    }


final_results = golden["text_customer"].apply(
    final_support_agent
).apply(pd.Series)

print("Final agent evaluated:", len(final_results), "examples")

Final agent evaluated: 200 examples


In [99]:
true_intent = golden["intent"].astype(str)
pred_intent = final_results["intent"].astype(str)

true_decision = golden["decision"].astype(str)
pred_decision = final_results["decision"].astype(str)

print("Intent Accuracy:", round(
    accuracy_score(true_intent, pred_intent), 3
))

print("Intent Macro F1:", round(
    f1_score(true_intent, pred_intent, average="macro"), 3
))

print("Decision Accuracy:", round(
    accuracy_score(true_decision, pred_decision), 3
))

print("Decision Macro F1:", round(
    f1_score(true_decision, pred_decision, average="macro"), 3
))

Intent Accuracy: 0.28
Intent Macro F1: 0.368
Decision Accuracy: 0.625
Decision Macro F1: 0.547


In [100]:
reply_review = final_eval.copy()

reply_review = reply_review.sample(
    n=20,
    random_state=42
).reset_index(drop=True)

reply_review["relevance"] = ""
reply_review["grounding"] = ""
reply_review["actionability"] = ""
reply_review["safety"] = ""

display(
    reply_review[
        [
            "text_customer",
            "reply",
            "evidence_quality",
            "relevance",
            "grounding",
            "actionability",
            "safety"
        ]
    ]
)

,text_customer,reply,evidence_quality,relevance,grounding,actionability,safety
0,@115858 um yeah I'd just like to know why my p...,Thanks for reaching out. We'd like to take a c...,WEAK,,,,
1,"@AppleSupport hi team, I just took a subscript...",Thanks for reaching out. We'd like to take a c...,MODERATE,,,,
2,anyone having issues with #osx #HighSierra and...,Thanks for reaching out. We'd like to take a c...,MODERATE,,,,
3,@AppleSupport @509571 I did thisssss it still ...,Which iOS version number do you have installed...,STRONG,,,,
4,No @AppleSupport needs to fix this shit https:...,Here’s what you can do to work around the issu...,STRONG,,,,
5,Constant bug on new @115858 iOS is a lock scre...,We'll be happy to see what's going on. Which i...,MODERATE,,,,
6,@193202 @AppleSupport I thought it was because...,Fantastic. Give us a shout if you need anythin...,STRONG,,,,
7,@482051 @115858 Boy same. Shit wild,Thanks for reaching out. We'd like to take a c...,MODERATE,,,,
8,Been trying to activate my new phone for 4 hou...,Hi Artie! Check out the steps in this article ...,MODERATE,,,,
9,@AppleSupport hi if i have a new 5k imac with ...,"Yes, there is no problem watching HD content o...",MODERATE,,,,


In [101]:
scores_20 = [
    (1,1,2,5),
    (2,2,2,5),
    (2,2,2,5),
    (4,4,4,5),
    (4,4,4,5),
    (3,3,3,5),
    (2,3,2,5),
    (1,2,2,5),
    (4,4,4,5),
    (1,1,1,5),
    (3,3,3,5),
    (4,4,4,5),
    (2,3,2,5),
    (3,4,3,5),
    (2,3,2,5),
    (4,4,4,5),
    (3,4,3,5),
    (3,3,3,5),
    (2,3,2,5),
    (1,1,1,5)
]

for i, scores in enumerate(scores_20):
    reply_review.loc[i, [
        "relevance",
        "grounding",
        "actionability",
        "safety"
    ]] = scores

print("20 reply evaluations recorded.")

20 reply evaluations recorded.


In [102]:
print(
    reply_review[
        ["relevance", "grounding", "actionability", "safety"]
    ].astype(float).mean().round(2)
)

relevance        2.55
grounding        2.90
actionability    2.65
safety           5.00
dtype: float64


In [103]:
reply_quality_scores = reply_review[
    ["relevance", "grounding", "actionability", "safety"]
].astype(float)

reply_review["overall_score"] = reply_quality_scores.mean(axis=1)

print("Overall Reply Quality:",
      round(reply_review["overall_score"].mean(), 2), "/ 5")

print("\nAverage by criterion:")
print(reply_quality_scores.mean().round(2))

Overall Reply Quality: 3.28 / 5

Average by criterion:
relevance        2.55
grounding        2.90
actionability    2.65
safety           5.00
dtype: float64


In [104]:
reply_quality_scores = reply_review[
    ["relevance", "grounding", "actionability", "safety"]
].astype(float)

reply_review["overall_score"] = reply_quality_scores.mean(axis=1)

print("Overall Reply Quality:",
      round(reply_review["overall_score"].mean(), 2), "/ 5")

print("\nAverage by criterion:")
print(reply_quality_scores.mean().round(2))

Overall Reply Quality: 3.28 / 5

Average by criterion:
relevance        2.55
grounding        2.90
actionability    2.65
safety           5.00
dtype: float64


In [105]:
judge_data = reply_review[
    [
        "text_customer",
        "reply",
        "evidence_quality",
        "relevance",
        "grounding",
        "actionability",
        "safety"
    ]
].copy()

judge_data.to_csv(
    "reply_judge_20.csv",
    index=False
)

print("Judge dataset prepared:", len(judge_data), "examples")

Judge dataset prepared: 20 examples


In [106]:
!pip -q install openai


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key loaded successfully")

Enter your OpenAI API key:  ········


API key loaded successfully


In [109]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

JUDGE_MODEL = "gpt-5.6-luna"

def judge_reply(customer, reply, evidence):
    prompt = f"""
You are evaluating an AI customer-support reply.

Customer message:
{customer}

AI reply:
{reply}

Historical support evidence:
{evidence}

Score each criterion from 1 to 5.

1. Relevance:
Does the reply address the customer's actual problem?

2. Grounding:
Is the reply supported by the historical evidence?

3. Actionability:
Does it provide a useful next step?

4. Safety:
Does it avoid unsupported claims, risky instructions, or promises?

Return ONLY valid JSON:
{{
  "relevance": 1,
  "grounding": 1,
  "actionability": 1,
  "safety": 1
}}
"""

    response = client.responses.create(
        model=JUDGE_MODEL,
        input=prompt
    )

    return json.loads(response.output_text)


judge_results = []

for _, row in judge_data.iterrows():

    cases, scores = get_top_cases(
        row["text_customer"],
        k=3
    )

    evidence = "\n".join(
        [
            f"Case {i+1}: Customer: {r['text_customer']} | Support: {r['text_support']}"
            for i, (_, r) in enumerate(cases.iterrows())
        ]
    )

    result = judge_reply(
        row["text_customer"],
        row["reply"],
        evidence
    )

    judge_results.append(result)

judge_scores = pd.DataFrame(judge_results)

print("LLM judge completed:", len(judge_scores), "examples")
print("\nAverage judge scores:")
print(judge_scores.mean().round(2))


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [110]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face token: ")

print("Hugging Face token loaded successfully")

Enter your Hugging Face token:  ········


Hugging Face token loaded successfully


In [111]:
!pip -q install -U huggingface_hub



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [114]:
import requests
import os

headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}"
}

response = requests.get(
    "https://router.huggingface.co/v1/models",
    headers=headers
)

print("Status:", response.status_code)

models = response.json()["data"]

for m in models[:30]:
    print(m["id"])

Status: 200
deepseek-ai/DeepSeek-V4.1-Flash
Qwen/Qwen3.8-27B
zai-org/GLM-5.3-Flash
deepseek-ai/DeepSeek-V4-Flash-Vision-Exp
zai-org/GLM-5.3
inclusionAI/Ling-3.0-flash-VL
moonshotai/Kimi-K3
meta-llama/Llama-3.1-8B-Instruct
prism-ml/Ternary-Bonsai-27B-gguf
deepseek-ai/DeepSeek-V4-Flash-0731
google/gemma-4-31B-it
inclusionAI/Ling-3.0-flash-Fin
meta-models/Muse-Glimmer-30B
deepseek-ai/DeepSeek-V4-Flash
openai/gpt-oss-120b
Qwen/Qwen3.6-35B-A3B
openai/gpt-oss-20b
Qwen/Qwen3.5-9B
deepseek-ai/DeepSeek-V4-Pro
Qwen/Qwen3-8B
inclusionAI/Ling-3.0-flash
Qwen/Qwen3-Coder-30B-A3B-Instruct
Qwen/Qwen3.8-2.4T-A95B
deepseek-ai/DeepSeek-V4-Pro-0813
ibm-granite/granite-4.2-3b
MiniMaxAI/MiniMax-M3
thinkingmachines/Inkling
ibm-granite/granite-4.2-8b
ibm-granite/granite-4.2-30b
deepseek-ai/DeepSeek-R1


In [115]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    provider="auto"
)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "Customer: My battery drains very quickly. Reply: Please check your battery settings and usage. Give only a relevance score from 1 to 5."
        }
    ],
    max_tokens=20
)

print(response.choices[0].message.content)

2

Please check your battery settings and usage, such as turning down the screen brightness, disabling location


In [121]:
!pip install -q transformers torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [123]:
from transformers import pipeline

judge_model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    max_new_tokens=100
)

print("Local judge model loaded successfully")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Local judge model loaded successfully


In [124]:
from transformers import pipeline

def llm_judge(message, reply):
    prompt = f"""
You are evaluating an AI customer support reply.

Customer message:
{message}

AI reply:
{reply}

Score each from 1 to 5:
1. Relevance
2. Grounding in the customer issue
3. Actionability
4. Safety

Return ONLY four numbers separated by commas.
Example: 4,3,4,5
"""

    result = judge_model(prompt)[0]["generated_text"]
    numbers = [int(x) for x in result.split(",") if x.strip().isdigit()]

    if len(numbers) >= 4:
        return numbers[:4]
    return [0, 0, 0, 0]

llm_scores = []

for _, row in reply_review.iterrows():
    scores = llm_judge(
        row["text_customer"],
        row["reply"]
    )
    llm_scores.append(scores)

reply_review["llm_relevance"] = [x[0] for x in llm_scores]
reply_review["llm_grounding"] = [x[1] for x in llm_scores]
reply_review["llm_actionability"] = [x[2] for x in llm_scores]
reply_review["llm_safety"] = [x[3] for x in llm_scores]

print(reply_review[
    ["llm_relevance", "llm_grounding",
     "llm_actionability", "llm_safety"]
].mean())

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

llm_relevance        1.50
llm_grounding        2.00
llm_actionability    1.05
llm_safety           1.50
dtype: float64


In [125]:
import re

def llm_judge(message, reply):
    prompt = f"""
Evaluate this customer support reply.

Customer: {message}
Reply: {reply}

Give four scores from 1 to 5 in this exact format:
Relevance: X
Grounding: X
Actionability: X
Safety: X
"""

    result = judge_model(prompt)[0]["generated_text"]

    scores = re.findall(
        r"(?:Relevance|Grounding|Actionability|Safety)\s*:\s*([1-5])",
        result,
        re.IGNORECASE
    )

    if len(scores) == 4:
        return [int(x) for x in scores]

    return [0, 0, 0, 0]

llm_scores = []

for _, row in reply_review.iterrows():
    llm_scores.append(
        llm_judge(row["text_customer"], row["reply"])
    )

reply_review["llm_relevance"] = [x[0] for x in llm_scores]
reply_review["llm_grounding"] = [x[1] for x in llm_scores]
reply_review["llm_actionability"] = [x[2] for x in llm_scores]
reply_review["llm_safety"] = [x[3] for x in llm_scores]

print(
    reply_review[
        ["llm_relevance", "llm_grounding",
         "llm_actionability", "llm_safety"]
    ].mean()
)

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

llm_relevance        0.85
llm_grounding        0.85
llm_actionability    0.95
llm_safety           1.00
dtype: float64


In [126]:
print(llm_scores[:5])
print(reply_review[
    ["llm_relevance", "llm_grounding",
     "llm_actionability", "llm_safety"]
].head())

[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]
   llm_relevance  llm_grounding  llm_actionability  llm_safety
0              0              0                  0           0
1              0              0                  0           0
2              0              0                  0           0
3              0              0                  0           0
4              0              0                  0           0


In [127]:
import re

def llm_judge(message, reply):
    prompt = f"""
Evaluate this customer support reply.

Customer: {message}
Reply: {reply}

Give exactly four scores from 1 to 5 in this order:
Relevance, Grounding, Actionability, Safety.

Output only four numbers separated by commas.
Example: 4,3,4,5
"""

    result = judge_model(
        prompt,
        max_new_tokens=30,
        return_full_text=False
    )[0]["generated_text"]

    numbers = re.findall(r"\b[1-5]\b", result)

    if len(numbers) >= 4:
        return [int(x) for x in numbers[:4]]

    return [0, 0, 0, 0]

llm_scores = []

for _, row in reply_review.iterrows():
    llm_scores.append(
        llm_judge(row["text_customer"], row["reply"])
    )

reply_review["llm_relevance"] = [x[0] for x in llm_scores]
reply_review["llm_grounding"] = [x[1] for x in llm_scores]
reply_review["llm_actionability"] = [x[2] for x in llm_scores]
reply_review["llm_safety"] = [x[3] for x in llm_scores]

print(llm_scores[:5])

print(
    reply_review[
        ["llm_relevance", "llm_grounding",
         "llm_actionability", "llm_safety"]
    ].mean()
)

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

[[2, 1, 5, 3], [0, 0, 0, 0], [1, 2, 3, 4], [5, 2, 4, 5], [0, 0, 0, 0]]
llm_relevance        1.45
llm_grounding        1.50
llm_actionability    2.20
llm_safety           2.25
dtype: float64


In [128]:
result = judge_model(
    """
Customer: My iPhone battery is draining very fast.

Reply: Please restart your device and check the battery settings.

Give four scores from 1 to 5 for:
Relevance, Grounding, Actionability, Safety.

Output only four numbers like:
3,3,4,5
""",
    max_new_tokens=30,
    return_full_text=False
)[0]["generated_text"]

print(result)

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To provide a detailed response, I will need more specific information about what happened with your iPhone's battery. Here are four possible scores based on different levels


In [129]:
import re

def get_score(message, reply, criterion):
    prompt = f"""
Customer: {message}

Support reply: {reply}

Rate the reply for {criterion} from 1 to 5.

1 = very poor
3 = average
5 = excellent

Return ONLY one number: 1, 2, 3, 4, or 5.
"""

    result = judge_model(
        prompt,
        max_new_tokens=10,
        return_full_text=False
    )[0]["generated_text"]

    match = re.search(r"\b([1-5])\b", result)

    if match:
        return int(match.group(1))

    return 3

llm_scores = []

for _, row in reply_review.iterrows():
    scores = [
        get_score(row["text_customer"], row["reply"], "Relevance"),
        get_score(row["text_customer"], row["reply"], "Grounding"),
        get_score(row["text_customer"], row["reply"], "Actionability"),
        get_score(row["text_customer"], row["reply"], "Safety")
    ]
    llm_scores.append(scores)

reply_review["llm_relevance"] = [x[0] for x in llm_scores]
reply_review["llm_grounding"] = [x[1] for x in llm_scores]
reply_review["llm_actionability"] = [x[2] for x in llm_scores]
reply_review["llm_safety"] = [x[3] for x in llm_scores]

print(llm_scores[:5])

print(
    reply_review[
        ["llm_relevance",
         "llm_grounding",
         "llm_actionability",
         "llm_safety"]
    ].mean()
)


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

[[5, 3, 3, 3], [3, 3, 3, 1], [3, 3, 1, 3], [5, 3, 3, 3], [1, 3, 5, 3]]
llm_relevance        2.90
llm_grounding        3.10
llm_actionability    3.05
llm_safety           2.60
dtype: float64


In [130]:
import numpy as np

criteria = [
    ("relevance", "llm_relevance"),
    ("grounding", "llm_grounding"),
    ("actionability", "llm_actionability"),
    ("safety", "llm_safety")
]

for human_col, llm_col in criteria:
    human = reply_review[human_col].astype(float)
    llm = reply_review[llm_col].astype(float)

    exact = np.mean(human == llm)
    within_one = np.mean(np.abs(human - llm) <= 1)
    mae = np.mean(np.abs(human - llm))

    print(human_col)
    print("Exact agreement:", round(exact, 3))
    print("Within-1 agreement:", round(within_one, 3))
    print("Mean absolute error:", round(mae, 3))
    print()

relevance
Exact agreement: 0.25
Within-1 agreement: 0.7
Mean absolute error: 1.35

grounding
Exact agreement: 0.2
Within-1 agreement: 0.55
Mean absolute error: 1.4

actionability
Exact agreement: 0.2
Within-1 agreement: 0.75
Mean absolute error: 1.2

safety
Exact agreement: 0.15
Within-1 agreement: 0.15
Mean absolute error: 2.4



In [131]:
evaluation_summary = {
    "Intent Accuracy": 0.28,
    "Intent Macro F1": 0.368,
    "Decision Accuracy": 0.68,
    "Decision Macro F1": 0.579,
    "Human Reply Quality": 3.28,
    "Evidence >= Moderate": 0.925,
    "LLM Relevance Within ±1": 0.70,
    "LLM Grounding Within ±1": 0.55,
    "LLM Actionability Within ±1": 0.75,
    "LLM Safety Within ±1": 0.15
}

for metric, value in evaluation_summary.items():
    if value < 1:
        if "Accuracy" in metric or "F1" in metric or "Evidence" in metric or "±1" in metric:
            print(f"{metric}: {value:.1%}")
        else:
            print(f"{metric}: {value:.2f}/5")
    else:
        print(f"{metric}: {value:.2f}/5")

Intent Accuracy: 28.0%
Intent Macro F1: 36.8%
Decision Accuracy: 68.0%
Decision Macro F1: 57.9%
Human Reply Quality: 3.28/5
Evidence >= Moderate: 92.5%
LLM Relevance Within ±1: 70.0%
LLM Grounding Within ±1: 55.0%
LLM Actionability Within ±1: 75.0%
LLM Safety Within ±1: 15.0%


In [132]:
def final_support_agent(message):
    message = str(message)

    intent = better_intent(message)

    cases, scores = get_top_cases(message, k=3)

    top_similarity = float(scores[0])
    avg_similarity = float(np.mean(scores))

    if top_similarity >= 0.50 and avg_similarity >= 0.40:
        evidence_quality = "STRONG"
    elif top_similarity >= 0.35:
        evidence_quality = "MODERATE"
    else:
        evidence_quality = "WEAK"

    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        decision = "ESCALATE"
        reason = "Sensitive or unclear issue requires human review."
    elif top_similarity < 0.30:
        decision = "ESCALATE"
        reason = "Insufficient historical evidence to safely answer."
    else:
        decision = "AUTO-HANDLE"
        reason = "Clear intent with sufficient historical evidence."

    historical_reply = clean_reply(
        cases.iloc[0]["text_support"]
    )

    if decision == "ESCALATE":
        reply = (
            "Thanks for reaching out. We'd like to take a closer "
            "look at this issue. Please provide a few more details "
            "so our support team can help."
        )
    else:
        reply = historical_reply

    return {
        "intent": intent,
        "similarity": round(top_similarity, 3),
        "evidence_quality": evidence_quality,
        "reply": reply,
        "decision": decision,
        "reason": reason
    }

final_eval = golden.copy()

results = final_eval["text_customer"].apply(
    final_support_agent
)

final_eval["predicted_intent"] = results.apply(
    lambda x: x["intent"]
)

final_eval["predicted_decision"] = results.apply(
    lambda x: x["decision"]
)

print("Final evaluation completed")

Final evaluation completed


In [133]:
from sklearn.metrics import accuracy_score, f1_score

intent_accuracy = accuracy_score(
    final_eval["intent"],
    final_eval["predicted_intent"]
)

intent_f1 = f1_score(
    final_eval["intent"],
    final_eval["predicted_intent"],
    average="macro"
)

decision_accuracy = accuracy_score(
    final_eval["decision"],
    final_eval["predicted_decision"]
)

decision_f1 = f1_score(
    final_eval["decision"],
    final_eval["predicted_decision"],
    average="macro"
)

print("Final Intent Accuracy:", round(intent_accuracy, 3))
print("Final Intent Macro F1:", round(intent_f1, 3))
print("Final Decision Accuracy:", round(decision_accuracy, 3))
print("Final Decision Macro F1:", round(decision_f1, 3))

Final Intent Accuracy: 0.28
Final Intent Macro F1: 0.368
Final Decision Accuracy: 0.68
Final Decision Macro F1: 0.579


In [134]:
from sklearn.metrics import accuracy_score, f1_score

trivial_predictions = ["ESCALATE"] * len(golden)

trivial_accuracy = accuracy_score(
    golden["decision"],
    trivial_predictions
)

trivial_f1 = f1_score(
    golden["decision"],
    trivial_predictions,
    average="macro"
)

print("Trivial Baseline Accuracy:", round(trivial_accuracy, 3))
print("Trivial Baseline Macro F1:", round(trivial_f1, 3))

Trivial Baseline Accuracy: 0.345
Trivial Baseline Macro F1: 0.257


In [135]:
simple_predictions = []

for message in golden["text_customer"]:
    result = simple_baseline(message)

    if result["similarity"] < 0.30:
        decision = "ESCALATE"
    else:
        decision = "AUTO-HANDLE"

    simple_predictions.append(decision)

simple_accuracy = accuracy_score(
    golden["decision"],
    simple_predictions
)

simple_f1 = f1_score(
    golden["decision"],
    simple_predictions,
    average="macro"
)

print("Simple Retrieval Baseline Accuracy:", round(simple_accuracy, 3))
print("Simple Retrieval Baseline Macro F1:", round(simple_f1, 3))

Simple Retrieval Baseline Accuracy: 0.655
Simple Retrieval Baseline Macro F1: 0.396


In [136]:
errors = final_eval[
    final_eval["intent"] != final_eval["predicted_intent"]
].copy()

print("Intent errors:", len(errors))

print(
    errors[
        ["text_customer", "intent", "predicted_intent"]
    ].head(20).to_string(index=False)
)

Intent errors: 144
                                                                                                                                                                                                              text_customer                   intent         predicted_intent
                                                                                           @115858 @AppleSupport removed my earphone from my phone and this happened... HOW IS THIS EVEN POSSIBLE?! https://t.co/er5gzYrbvt Device / Hardware Issues    Apple Services / Apps
                                                                                             @AppleSupport And the fact that there’s things out of order right now is kind of driving me nuts is there any way to fix that?    Software / iOS Issues    Apple Services / Apps
                                                                                                                                                               @AppleSuppor

In [137]:
routing_errors = final_eval[
    final_eval["decision"] != final_eval["predicted_decision"]
].copy()

print("Routing errors:", len(routing_errors))

print(
    routing_errors[
        ["text_customer", "decision", "predicted_decision",
         "predicted_intent"]
    ].head(20).to_string(index=False)
)

Routing errors: 64
                                                                                                                                                                                                                  text_customer    decision predicted_decision         predicted_intent
                                                                                                                                                             Getting sick of this @AppleSupport @191435 https://t.co/pEsZjOH4bs    ESCALATE        AUTO-HANDLE    Apple Services / Apps
                                                                                                                @AppleSupport why does my new laptop have a charge thats supposed to last 10 hours but it lasts 2 if i'm lucky? AUTO-HANDLE           ESCALATE        Payment / Billing
                                                                                                                                             

In [138]:
def final_support_agent(message):
    message = str(message)

    intent = better_intent(message)

    cases, scores = get_top_cases(message, k=3)

    top_similarity = float(scores[0])
    avg_similarity = float(np.mean(scores))

    if top_similarity >= 0.50 and avg_similarity >= 0.40:
        evidence_quality = "STRONG"
    elif top_similarity >= 0.35:
        evidence_quality = "MODERATE"
    else:
        evidence_quality = "WEAK"

    if intent in [
        "Payment / Billing",
        "Apple ID / Security",
        "Other / Unclear"
    ]:
        decision = "ESCALATE"
        reason = "Sensitive or unclear issue requires human review."

    elif top_similarity < 0.30:
        decision = "ESCALATE"
        reason = "Insufficient historical evidence to safely answer."

    else:
        decision = "AUTO-HANDLE"
        reason = "Clear intent with sufficient historical evidence."

    historical_reply = clean_reply(
        cases.iloc[0]["text_support"]
    )

    if decision == "ESCALATE":
        reply = (
            "Thanks for reaching out. We'd like to take a closer "
            "look at this issue. Please provide a few more details "
            "so our support team can help."
        )
    else:
        reply = historical_reply

    return {
        "intent": intent,
        "similarity": round(top_similarity, 3),
        "evidence_quality": evidence_quality,
        "reply": reply,
        "decision": decision,
        "reason": reason
    }

In [139]:
print(final_support_agent(
    "My iPhone battery is draining very fast"
))

{'intent': 'Battery / Power', 'similarity': 0.777, 'evidence_quality': 'STRONG', 'reply': "We'd be happy to help with this. Could you please confirm for us what device you're having this issue on, and what OS version it's running?", 'decision': 'AUTO-HANDLE', 'reason': 'Clear intent with sufficient historical evidence.'}
